In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from dsm_ga import load_data, GAConfig, run_ga

## Loading DSM from CSV files

In [ ]:
freq_csv_path   = "consolidation_potential_DSM_scenario1_interaction_frequency.csv"
consol_csv_path = "consolidation_potential_DSM_scenario1_SA4_FTE2_2026-08-04-12h27.csv"

DSM_freq, DSM_consol, units, DSM_header_list = load_data(freq_csv_path, consol_csv_path)
DSM = DSM_freq   # alias used by visualisation cells below
NUMBER_OF_DSM_ELEMENTS = len(units)
csv_file = consol_csv_path.replace(".csv", "")
print(f"Loaded {NUMBER_OF_DSM_ELEMENTS} units")

In [ ]:
def style_freq(df):
    vals = df.values.astype(float).copy()
    np.fill_diagonal(vals, np.nan)
    vmax = np.nanmax(vals)
    cmap = plt.cm.YlOrRd
    styled = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            if i == j:
                styled.iloc[i, j] = 'background-color: #f0f0f0'
            elif df.iloc[i, j] > 0:
                intensity = df.iloc[i, j] / vmax
                colour = mcolors.rgb2hex(cmap(intensity))
                styled.iloc[i, j] = f'background-color: {colour}'
    return styled

DSM_freq.style.apply(style_freq, axis=None)

In [ ]:
def style_consol(df):
    styled = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            if i == j:
                styled.iloc[i, j] = 'background-color: #f0f0f0'
            elif df.iloc[i, j] == 1:
                styled.iloc[i, j] = 'background-color: #6fa8dc'
    return styled

DSM_consol.style.apply(style_consol, axis=None)

# Definition of 3 term Fitness function with consolidation

### Metric used is the Minimum Description length Principle defined in: Tian-Li Yu, Ali A. Yassine, David E. Goldberg paper - Equation 3 - MDL clustering metric
### fDSM(M) = (1 - alpha - beta) * (Nc * log(Nn) + (log(Nn)*SUM_i_to_Nc(CLi)) + (alpha * [|S1| * (2log(Nn + 1))] + beta[|S2| * (2log(Nn + 1))])

##### Nc = number of clusters in the DSM
##### Nn = number of rows/columns in the DSM
##### CLi = number of nodes in cluster i
##### S1 = Type 1 error: every relationship not included in a cluster
##### S2 = Type 2 error: a node in a cluster with no directional relationships to other nodes in that cluster
##### S3 = Type 3 error: nodes that should be consolidated but are not

# Optimization Algorithm
### Parameters

In [ ]:
config = GAConfig(
    alpha=0.15,              # weight: type 1 error (connections across cluster boundaries)
    beta=0.05,               # weight: type 2 error (no connection within cluster)
    gamma=0.15,              # weight: type 3 error (consolidation potential across clusters)
    delta=0.20,              # weight: cluster size imbalance penalty
    max_clusters=10,
    target_clusters=10,
    consolidation_mode='once',   # 'once' | 'directional'
    population_size=200,
    n_generations=400,
    cxpb=0.5,
    mutpb=0.1,
    tournsize=5,
)

### Optimization loop

In [ ]:
import os
checkpoint_file = csv_file + "_checkpoint.pkl"
try:
    os.remove(checkpoint_file)
    print("Deleted existing checkpoint")
except FileNotFoundError:
    print("No checkpoint found")

In [ ]:
best, logbook, final_fitness = run_ga(
    DSM_freq, DSM_consol, config,
    checkpoint_file=checkpoint_file,
    verbose=True,
)
print(f"Best solution: {best}")
print(f"Final fitness: {final_fitness:.4f}")

# Export optimized model to Excel

In [ ]:
group_assignments = list(best)
sorted_indices = sorted(range(NUMBER_OF_DSM_ELEMENTS), key=lambda i: group_assignments[i])
sorted_units  = [DSM_header_list[i] for i in sorted_indices]
sorted_groups = [group_assignments[i] for i in sorted_indices]

DSM_reordered = DSM.iloc[sorted_indices].iloc[:, sorted_indices].copy()
DSM_reordered.index   = sorted_units
DSM_reordered.columns = sorted_units

DSM_consol_reordered = DSM_consol.iloc[sorted_indices].iloc[:, sorted_indices].copy()
DSM_consol_reordered.index   = sorted_units
DSM_consol_reordered.columns = sorted_units

unique_clusters = sorted(set(sorted_groups))
palette = plt.cm.Set2(np.linspace(0, 0.8, len(unique_clusters)))
cluster_color_map = {c: mcolors.rgb2hex(palette[i]) for i, c in enumerate(unique_clusters)}

cluster_starts = {}
cluster_ends   = {}
for idx, group in enumerate(sorted_groups):
    if group not in cluster_starts:
        cluster_starts[group] = idx
    cluster_ends[group] = idx

COLOUR_ACHIEVED = '#93c47d'
COLOUR_MISSED   = '#e06666'

def _cell_style(i, j, has_consol):
    parts = []
    same_cluster = sorted_groups[i] == sorted_groups[j]
    if has_consol and same_cluster:
        parts.append(f'background-color: {COLOUR_ACHIEVED}')
    elif has_consol and not same_cluster:
        parts.append(f'background-color: {COLOUR_MISSED}')
    if same_cluster:
        c     = sorted_groups[i]
        color = cluster_color_map[c]
        if i == cluster_starts[c]: parts.append(f'border-top: 2px solid {color}')
        if i == cluster_ends[c]:   parts.append(f'border-bottom: 2px solid {color}')
        if j == cluster_starts[c]: parts.append(f'border-left: 2px solid {color}')
        if j == cluster_ends[c]:   parts.append(f'border-right: 2px solid {color}')
    return '; '.join(parts)

def style_func(df):
    styled = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            styled.iloc[i, j] = _cell_style(i, j, DSM_consol_reordered.iloc[i, j] == 1)
    return styled

def style_func_consol(df):
    styled = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            styled.iloc[i, j] = _cell_style(i, j, df.iloc[i, j] == 1)
    return styled

DSM_styled = DSM_reordered.style.apply(style_func, axis=None)
DSM_styled

In [ ]:
# Publication-quality DSM figure
def plot_dsm_paper(freq_df, consol_df, units, groups, title=None, dpi=300, save_path=None):
    import matplotlib.patches as patches
    import numpy as np
    n = len(units)
    unique_clusters = sorted(set(groups))

    c_start, c_end = {}, {}
    for idx, g in enumerate(groups):
        if g not in c_start: c_start[g] = idx
        c_end[g] = idx

    CONSOL_COLOR = [0.40, 0.63, 0.85, 1.0]
    DIAG_COLOR   = [0.88, 0.88, 0.88, 1.0]
    WHITE        = [1.00, 1.00, 1.00, 1.0]

    rgba   = np.tile(WHITE, (n, n, 1))
    freq   = freq_df.values.astype(float)
    consol = consol_df.values.astype(float)
    for i in range(n):
        for j in range(n):
            if i == j:
                rgba[i, j] = DIAG_COLOR
            elif consol[i, j] == 1:
                rgba[i, j] = CONSOL_COLOR

    cell_size = max(0.28, min(0.45, 10.0 / n))
    fig_size  = max(8, n * cell_size + 2.5)

    fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=dpi)
    ax.imshow(rgba, aspect='equal', interpolation='none', origin='upper')

    fontsize = max(4.5, min(8.0, 95.0 / n))
    for i in range(n):
        for j in range(n):
            if i != j and freq[i, j] > 0:
                on_blue = consol[i, j] == 1
                ax.text(j, i, str(int(freq[i, j])),
                        ha='center', va='center', fontsize=fontsize,
                        color='white' if on_blue else '#333333',
                        fontfamily='sans-serif')

    lw = max(1.2, min(2.0, 14.0 / n))
    for g in unique_clusters:
        s, e = c_start[g], c_end[g]
        rect = patches.Rectangle(
            (s - 0.5, s - 0.5), e - s + 1, e - s + 1,
            linewidth=lw, edgecolor='#111111', facecolor='none', zorder=4)
        ax.add_patch(rect)

    lbl_fs = max(5.5, min(8.5, 85.0 / n))
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(units, rotation=90, fontsize=lbl_fs, ha='center', fontfamily='sans-serif')
    ax.set_yticklabels(units, fontsize=lbl_fs, fontfamily='sans-serif')
    ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False,
                   left=True, labelleft=True, length=0)
    ax.xaxis.set_label_position('top')
    for spine in ax.spines.values():
        spine.set_visible(False)

    clbl_fs = max(6, min(10, 100.0 / n))
    for g in unique_clusters:
        s, e = c_start[g], c_end[g]
        ax.text((s + e) / 2.0, (s + e) / 2.0, 'G' + str(g),
                ha='center', va='center', fontsize=clbl_fs,
                fontweight='bold', color='#222222', zorder=5,
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.75, linewidth=0))

    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', pad=14, fontfamily='sans-serif')

    from matplotlib.patches import Patch
    ax.legend(
        handles=[
            Patch(facecolor=CONSOL_COLOR[:3], edgecolor='none', label='Consolidation potential'),
            Patch(facecolor='white', edgecolor='#aaaaaa', linewidth=0.8, label='No consolidation potential'),
        ],
        loc='lower right', bbox_to_anchor=(1.0, -0.18),
        fontsize=8, frameon=True, framealpha=0.9, edgecolor='#cccccc', ncol=2,
    )

    plt.tight_layout(pad=1.5)
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print('Saved: ' + save_path)
    plt.show()
    return fig


fig_dsm = plot_dsm_paper(
    DSM_reordered, DSM_consol_reordered,
    sorted_units, sorted_groups,
    title='DSM Optimized Grouping  |  Fitness: ' + str(round(final_fitness, 4)),
    dpi=300,
    save_path=csv_file + '_dsm_figure.png',
)


In [ ]:
df_optimized_grouping = pd.DataFrame({
    'Unit Name': sorted_units,
    'Cluster':   [f'Cluster {g}' for g in sorted_groups],
})
df_optimized_grouping

In [ ]:
DSM_consol_styled = DSM_consol_reordered.style.apply(style_func_consol, axis=None)

optimized_path = csv_file + '_optimized.xlsx'
with pd.ExcelWriter(optimized_path, engine='openpyxl') as writer:
    DSM_styled.to_excel(writer, sheet_name='dsm_optimized')
    DSM_consol_styled.to_excel(writer, sheet_name='dsm_consolidation')
    df_optimized_grouping.to_excel(writer, sheet_name='grouping')
    DSM_freq.to_excel(writer, sheet_name='interaction_frequency')
    DSM_consol.to_excel(writer, sheet_name='consolidation_potential')

print(f'Exported to {optimized_path}')